# <font color="#418FDE" size="6.5" uppercase>**MLP und Faltungen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Trainieren ein kleines Zwei-Schichten-MLP mit NumPy auf einfachen Klassifikationsdaten. 
- Vergleichen Mini-Batches, Shuffling, SGD, Momentum, Regularisierung und Early Stopping. 
- Implementieren grundlegende Faltungs- und Poolingoperationen und planen Tensorformen. 


## **1. MLP trainieren**

### **1.1. Kleine Datensätze**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_01.jpg?v=1787654570" width="250">



>* Kleine Daten machen MLP-Training nachvollziehbar
>* Entscheidungsgrenzen zeigen Vorteile verborgener Schichten

>* Datenqualität prägt kleine Trainingsdatensätze stark
>* Kleine Datensätze zeigen typische ML-Probleme

>* Lernverhalten schnell beobachten und deuten
>* Überanpassung durch Testdaten erkennen



In [ ]:
#@title Python-Code - Kleine Datensätze

# Wir trainieren ein kleines MLP mit NumPy.
# Kleine Daten machen Lernschritte gut sichtbar.
# Die Grafik zeigt Datenpunkte und Entscheidungsgrenze.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz bleibt übersichtlich und schnell.
features, labels = make_moons(n_samples=240, noise=0.22, random_state=42)
labels = labels.reshape(-1, 1)

# Die Aufteilung prüft Lernen auf unbekannten Punkten.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Diese Prüfung schützt vor unerwarteten Formen.
if X_train.shape[1] != 2 or y_train.shape[1] != 1:
    raise ValueError("Erwartet werden zwei Merkmale und ein Ziel.")

# Kleine zufällige Gewichte starten das MLP.
rng = np.random.default_rng(42)
W1 = rng.normal(0.0, 0.5, size=(2, 8))
b1 = np.zeros((1, 8))

# Die Ausgabeschicht liefert eine Klassenwahrscheinlichkeit.
W2 = rng.normal(0.0, 0.5, size=(8, 1))
b2 = np.zeros((1, 1))

# Diese Einstellungen halten das Training kurz.
learning_rate = 0.08
epochs = 80
train_losses = []

# Vorwärtsrechnung, Fehler und Rückwärtsrechnung wiederholen sich.
for epoch in range(epochs):
    hidden_linear = X_train @ W1 + b1
    hidden = np.tanh(hidden_linear)
    logits = hidden @ W2 + b2
    predictions = 1.0 / (1.0 + np.exp(-logits))

    error = predictions - y_train
    loss = -np.mean(
        y_train * np.log(predictions + 1e-8)
        + (1 - y_train) * np.log(1 - predictions + 1e-8)
    )
    train_losses.append(loss)

    grad_W2 = hidden.T @ error / len(X_train)
    grad_b2 = np.mean(error, axis=0, keepdims=True)
    hidden_error = (error @ W2.T) * (1 - hidden * hidden)
    grad_W1 = X_train.T @ hidden_error / len(X_train)

    grad_b1 = np.mean(hidden_error, axis=0, keepdims=True)
    W2 = W2 - learning_rate * grad_W2
    b2 = b2 - learning_rate * grad_b2
    W1 = W1 - learning_rate * grad_W1

    b1 = b1 - learning_rate * grad_b1

# Eine kleine Hilfsrechnung erzeugt Klassen aus Wahrscheinlichkeiten.
def predict_classes(data):
    hidden = np.tanh(data @ W1 + b1)
    probabilities = 1.0 / (1.0 + np.exp(-(hidden @ W2 + b2)))
    return (probabilities >= 0.5).astype(int)

# Genauigkeit vergleicht Vorhersagen mit echten Klassen.
train_accuracy = np.mean(predict_classes(X_train) == y_train)
test_accuracy = np.mean(predict_classes(X_test) == y_test)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsverlust: {train_losses[0]:.3f} -> {train_losses[-1]:.3f}")
print(f"Trainingsgenauigkeit: {train_accuracy:.2f}")
print(f"Testgenauigkeit: {test_accuracy:.2f}")

# Ein Gitter macht die gelernte Entscheidungsgrenze sichtbar.
x_min = X_train[:, 0].min() - 0.6
x_max = X_train[:, 0].max() + 0.6
y_min = X_train[:, 1].min() - 0.6

# Die Gitterpunkte decken den sichtbaren Bereich ab.
y_max = X_train[:, 1].max() + 0.6
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_classes = predict_classes(grid).reshape(xx.shape)

# Die Grafik verbindet Datenpunkte und Modellentscheidung.
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, grid_classes, levels=[-0.5, 0.5, 1.5], alpha=0.25)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), edgecolor="k", s=35)

ax.set_title("Kleines Zwei-Schichten-MLP auf kleinen Daten")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
plt.show()



### **1.2. Netzarchitektur**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_02.jpg?v=1787654568" width="250">



>* Eingabe, verborgene Schicht und Ausgabe
>* Nichtlinearität ermöglicht flexible Klassenvorhersagen

>* Eingabegröße folgt der Merkmalsanzahl
>* Verborgene Größe steuert Ausdruckskraft und Überanpassung

>* Ausgabe passend zur Klassifikationsaufgabe wählen
>* Einfaches MLP macht Trainingsdynamik nachvollziehbar



In [ ]:
#@title Python-Code - Netzarchitektur

# Dieses Beispiel zeigt eine kleine MLP-Netzarchitektur.
# Tensorformen machen den Datenfluss im Netz sichtbar.
# Am Ende sehen wir eine einfache Entscheidungsgrenze.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir erzeugen kleine, nichtlinear trennbare Klassifikationsdaten.
features, labels = make_moons(n_samples=300, noise=0.18, random_state=42)

# Die Aufteilung hält Trainings- und Testdaten sauber getrennt.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung wird nur auf den Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Diese Größen definieren die Architektur des Zwei-Schichten-MLP.
input_size = X_train.shape[1]
hidden_size = 8
output_size = 1

# Eine kurze Prüfung verhindert unpassende Matrixformen.
if input_size != 2:
    raise ValueError("Dieses Beispiel erwartet genau zwei Eingabemerkmale.")

# Kleine Zufallsgewichte starten das Lernen deterministisch.
rng = np.random.default_rng(42)
W1 = rng.normal(0.0, 0.5, size=(input_size, hidden_size))
b1 = np.zeros((1, hidden_size))

# Die Ausgabeschicht liefert eine Wahrscheinlichkeit pro Datenpunkt.
W2 = rng.normal(0.0, 0.5, size=(hidden_size, output_size))
b2 = np.zeros((1, output_size))

# Diese Funktionen bilden Aktivierungen und Vorhersagen.
def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))

# Der Vorwärtsdurchlauf zeigt die Architektur als Matrixkette.
def forward_pass(X):
    hidden_linear = X @ W1 + b1
    hidden_activation = np.tanh(hidden_linear)
    output_linear = hidden_activation @ W2 + b2
    return hidden_activation, sigmoid(output_linear)

# Wir trainieren bewusst kurz mit einfachem Gradientenabstieg.
learning_rate = 0.08
epochs = 100
n_train = X_train.shape[0]

# Jede Epoche aktualisiert beide Schichten gemeinsam.
for epoch in range(epochs):
    hidden, predictions = forward_pass(X_train)
    error = predictions - y_train.reshape(-1, 1)
    dW2 = hidden.T @ error / n_train

    db2 = np.mean(error, axis=0, keepdims=True)
    hidden_error = (error @ W2.T) * (1.0 - hidden * hidden)
    dW1 = X_train.T @ hidden_error / n_train
    db1 = np.mean(hidden_error, axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

# Die Testgenauigkeit prüft die gelernte Architektur.
_, test_probabilities = forward_pass(X_test)
test_predictions = (test_probabilities >= 0.5).astype(int).ravel()
test_accuracy = np.mean(test_predictions == y_test)

# Wir geben nur die wichtigsten Architekturinformationen aus.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Architektur: {input_size} Eingaben -> {hidden_size} verborgen -> {output_size} Ausgabe")
print(f"Form W1: {W1.shape}, Form W2: {W2.shape}")
print(f"Testgenauigkeit: {test_accuracy:.2f}")

# Ein Gitter macht die gelernte Entscheidungsgrenze sichtbar.
x_min = X_train[:, 0].min() - 0.5
x_max = X_train[:, 0].max() + 0.5
y_min = X_train[:, 1].min() - 0.5

# Die zweite Achse nutzt dieselbe Skalierung wie die Daten.
y_max = X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160))
grid = np.c_[xx.ravel(), yy.ravel()]

# Für jeden Gitterpunkt berechnet das MLP eine Klasse.
_, grid_probabilities = forward_pass(grid)
grid_classes = (grid_probabilities.reshape(xx.shape) >= 0.5).astype(int)

# Die Grafik verbindet Architektur, Datenfluss und Vorhersage.
fig, ax = plt.subplots(figsize=(6, 4))
ax.contourf(xx, yy, grid_classes, levels=[-0.5, 0.5, 1.5], alpha=0.25)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=25, edgecolor="k")

# Achsen und Titel beschreiben die skalierten Eingabemerkmale.
ax.set_title("Zwei-Schichten-MLP: gelernte Entscheidungsgrenze")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
plt.show()



### **1.3. Trainingsschleife verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_03.jpg?v=1787654572" width="250">



>* Vorhersagen prüfen und Gewichte verbessern
>* Wiederholungen lassen das MLP Muster erkennen

>* Fehler berechnen und per Backpropagation zuordnen
>* Lernrate steuert viele kleine Parameterkorrekturen

>* Validierungsdaten zeigen Überanpassung frühzeitig.
>* Training soll auch neue Daten meistern.



In [ ]:
#@title Python-Code - Trainingsschleife verstehen

# Wir trainieren ein kleines MLP mit NumPy.
# Die Schleife zeigt Vorwärtslauf und Rückwärtspropagation.
# Am Ende sinkt der Fehler sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz macht die Trainingsschleife überschaubar.
features, labels = make_moons(
    n_samples=300,
    noise=0.18,
    random_state=42,
)

# Die Aufteilung prüft Lernen auf unbekannten Beispielen.
X_train, X_val, y_train, y_val = train_test_split(
    features,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42,
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Diese Prüfung verhindert unpassende Eingabeformen.
if X_train.shape[1] != 2:
    raise ValueError("Erwartet werden genau zwei Eingabemerkmale.")

# Zufällige Startgewichte sind klein und reproduzierbar.
rng = np.random.default_rng(42)
hidden_size = 8
learning_rate = 0.08

W1 = rng.normal(0.0, 0.5, size=(2, hidden_size))
b1 = np.zeros((1, hidden_size))
W2 = rng.normal(0.0, 0.5, size=(hidden_size, 1))
b2 = np.zeros((1, 1))

# Diese Listen speichern den Verlauf pro Epoche.
train_losses = []
val_losses = []
epochs = 80

# Die Trainingsschleife wiederholt Vorhersage, Fehler und Update.
for epoch in range(epochs):
    hidden_linear = X_train @ W1 + b1
    hidden = np.tanh(hidden_linear)
    logits = hidden @ W2 + b2
    predictions = 1.0 / (1.0 + np.exp(-logits))

    y_column = y_train.reshape(-1, 1)
    error = predictions - y_column
    train_loss = np.mean(
        -(y_column * np.log(predictions + 1e-8)
        + (1 - y_column) * np.log(1 - predictions + 1e-8))
    )

    sample_count = X_train.shape[0]
    d_logits = error / sample_count
    d_W2 = hidden.T @ d_logits
    d_b2 = np.sum(d_logits, axis=0, keepdims=True)

    d_hidden = d_logits @ W2.T
    d_hidden_linear = d_hidden * (1 - hidden ** 2)
    d_W1 = X_train.T @ d_hidden_linear
    d_b1 = np.sum(d_hidden_linear, axis=0, keepdims=True)

    W1 = W1 - learning_rate * d_W1
    b1 = b1 - learning_rate * d_b1
    W2 = W2 - learning_rate * d_W2
    b2 = b2 - learning_rate * d_b2

    val_hidden = np.tanh(X_val @ W1 + b1)
    val_logits = val_hidden @ W2 + b2
    val_predictions = 1.0 / (1.0 + np.exp(-val_logits))

    val_column = y_val.reshape(-1, 1)
    val_loss = np.mean(
        -(val_column * np.log(val_predictions + 1e-8)
        + (1 - val_column) * np.log(1 - val_predictions + 1e-8))
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

# Genauigkeit zeigt die Klassifikationsleistung nach dem Training.
train_classes = (predictions >= 0.5).astype(int).ravel()
val_classes = (val_predictions >= 0.5).astype(int).ravel()
train_accuracy = np.mean(train_classes == y_train)
val_accuracy = np.mean(val_classes == y_val)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Startfehler Training: {train_losses[0]:.3f}")
print(f"Endfehler Training: {train_losses[-1]:.3f}")
print(f"Validierungsgenauigkeit: {val_accuracy:.3f}")

# Die Kurven machen den Lernfortschritt sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label="Training")
ax.plot(val_losses, label="Validierung")
ax.set_title("Fehlerverlauf einer MLP-Trainingsschleife")

ax.set_xlabel("Epoche")
ax.set_ylabel("Kreuzentropie-Fehler")
ax.legend()
plt.show()



## **2. Training verbessern**

### **2.1. Mini Batches verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_01.jpg?v=1787654583" width="250">



>* Mini-Batches balancieren Stabilität und Geschwindigkeit
>* Kleine Stichproben ermöglichen effiziente Gewichtsupdates

>* Mini-Batches bringen hilfreiche Zufälligkeit ins Training
>* Rückmeldung erfolgt in überschaubaren Lernschritten

>* Shuffling verhindert verzerrte Mini-Batches
>* Batch-Größe balanciert Effizienz und Vielfalt



In [ ]:
#@title Python-Code - Mini Batches verstehen

# Dieses Beispiel zeigt Mini-Batches beim Training.
# Batch-Größen verändern die Anzahl der Updates.
# Die Grafik vergleicht glatte und schwankende Lernkurven.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sklearn

# Wir erzeugen kleine, gut kontrollierbare Klassifikationsdaten.
features, labels = make_classification(
    n_samples=600, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

# Die Aufteilung bleibt reproduzierbar und klassenweise ausgewogen.
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# Eine Eins-Spalte erlaubt einen Bias im linearen Modell.
train_design = np.c_[np.ones(len(train_features)), train_features]
test_design = np.c_[np.ones(len(test_features)), test_features]

# Diese Prüfung macht die erwartete Form sichtbar.
if train_design.shape[1] != 3:
    raise ValueError("Die Trainingsmatrix sollte drei Spalten haben.")

# Sigmoid wandelt lineare Werte in Wahrscheinlichkeiten um.
def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))

# Diese Funktion trainiert dasselbe Modell mit einer Batch-Größe.
def train_with_batch_size(batch_size, epochs=25, learning_rate=0.15):
    rng = np.random.default_rng(42)
    weights = np.zeros(train_design.shape[1])
    losses = []

    for epoch in range(epochs):
        order = rng.permutation(len(train_design))
        shuffled_x = train_design[order]
        shuffled_y = train_labels[order]

        for start in range(0, len(shuffled_x), batch_size):
            end = start + batch_size
            batch_x = shuffled_x[start:end]
            batch_y = shuffled_y[start:end]

            predictions = sigmoid(batch_x @ weights)
            gradient = batch_x.T @ (predictions - batch_y) / len(batch_x)
            weights = weights - learning_rate * gradient

        train_probabilities = sigmoid(train_design @ weights)
        clipped = np.clip(train_probabilities, 1e-7, 1 - 1e-7)
        loss = -np.mean(
            train_labels * np.log(clipped)
            + (1 - train_labels) * np.log(1 - clipped)
        )
        losses.append(loss)

    test_predictions = sigmoid(test_design @ weights) >= 0.5
    accuracy = np.mean(test_predictions == test_labels)
    return np.array(losses), accuracy

# Kleine Batches aktualisieren häufiger als große Batches.
small_losses, small_accuracy = train_with_batch_size(batch_size=8)
large_losses, large_accuracy = train_with_batch_size(batch_size=128)

# Die Ausgabe bleibt kurz und konzentriert.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsbeispiele: {len(train_design)}, Merkmale mit Bias: {train_design.shape[1]}")
print(f"Batch 8:  {int(np.ceil(len(train_design) / 8))} Updates pro Epoche")
print(f"Batch 128: {int(np.ceil(len(train_design) / 128))} Updates pro Epoche")
print(f"Testgenauigkeit Batch 8: {small_accuracy:.2f}")
print(f"Testgenauigkeit Batch 128: {large_accuracy:.2f}")

# Die Lernkurven zeigen den Effekt der Batch-Größe.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(small_losses) + 1), small_losses, label="Batch 8")
ax.plot(range(1, len(large_losses) + 1), large_losses, label="Batch 128")

# Achsen und Legende machen den Vergleich lesbar.
ax.set_title("Mini-Batches: Verlust pro Epoche")
ax.set_xlabel("Epoche")
ax.set_ylabel("Trainingsverlust")
ax.legend()

plt.show()



### **2.2. SGD mit Momentum**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_02.jpg?v=1787654579" width="250">



>* SGD lernt schnell mit verrauschten Mini-Batches
>* Momentum glättet Schritte durch gespeicherte Richtung

>* Momentum glättet Schwankungen und beschleunigt Lernen
>* Die Lernrate bleibt trotzdem entscheidend

>* Momentum glättet Mini-Batch-Schwankungen kontrolliert
>* Shuffling, Regularisierung und Early Stopping ergänzen



In [ ]:
#@title Python-Code - SGD mit Momentum

# Wir vergleichen SGD mit und ohne Momentum.
# Momentum glättet schwankende Mini-Batch-Schritte.
# Die Kurven zeigen stabilere Optimierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine, reproduzierbare Klassifikationsdaten.
features, labels = make_classification(
    n_samples=600, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

# Die Aufteilung bleibt stratifiziert und reproduzierbar.
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# Eine Bias-Spalte macht das lineare Modell einfacher.
train_design = np.c_[np.ones(train_features.shape[0]), train_features]
test_design = np.c_[np.ones(test_features.shape[0]), test_features]

# Diese Prüfung schützt vor unerwarteten Formproblemen.
if train_design.shape[1] != 3:
    raise ValueError("Die Datenform passt nicht zum Beispiel.")

# Die Sigmoidfunktion wandelt Werte in Wahrscheinlichkeiten um.
def sigmoid(values):
    clipped_values = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped_values))

# Diese Funktion berechnet den mittleren Log-Loss.
def log_loss(design, labels, weights):
    probabilities = sigmoid(design @ weights)
    probabilities = np.clip(probabilities, 1e-8, 1.0 - 1e-8)
    losses = -labels * np.log(probabilities)
    losses -= (1 - labels) * np.log(1 - probabilities)
    return float(np.mean(losses))

# Mini-Batches werden in jeder Epoche neu gemischt.
def train_sgd(use_momentum):
    rng = np.random.default_rng(42)
    weights = np.zeros(train_design.shape[1])
    velocity = np.zeros_like(weights)
    losses = []
    for epoch in range(60):
        order = rng.permutation(train_design.shape[0])
        for start in range(0, train_design.shape[0], 32):
            batch_index = order[start:start + 32]
            batch_x = train_design[batch_index]
            batch_y = train_labels[batch_index]
            prediction = sigmoid(batch_x @ weights)
            gradient = batch_x.T @ (prediction - batch_y) / len(batch_y)
            if use_momentum:
                velocity = 0.9 * velocity - 0.12 * gradient
                weights = weights + velocity
            else:
                weights = weights - 0.12 * gradient
        losses.append(log_loss(train_design, train_labels, weights))
    return weights, losses

# Beide Varianten starten gleich und sehen dieselben Mini-Batches.
plain_weights, plain_losses = train_sgd(use_momentum=False)
momentum_weights, momentum_losses = train_sgd(use_momentum=True)

# Genauigkeit zeigt, ob die Optimierung praktisch funktioniert.
plain_accuracy = np.mean((sigmoid(test_design @ plain_weights) >= 0.5) == test_labels)
momentum_accuracy = np.mean((sigmoid(test_design @ momentum_weights) >= 0.5) == test_labels)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit ohne Momentum: {plain_accuracy:.3f}")
print(f"Testgenauigkeit mit Momentum: {momentum_accuracy:.3f}")
print(f"Letzter Trainingsverlust ohne Momentum: {plain_losses[-1]:.3f}")
print(f"Letzter Trainingsverlust mit Momentum: {momentum_losses[-1]:.3f}")

# Die Lernkurven machen die Glättung sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plain_losses, label="SGD ohne Momentum")
ax.plot(momentum_losses, label="SGD mit Momentum")
ax.set_title("Trainingsverlust bei Mini-Batch-SGD")
ax.set_xlabel("Epoche")
ax.set_ylabel("Log-Loss")
ax.legend()
plt.show()



### **2.3. Regularisierung gegen Überanpassung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_03.jpg?v=1787654581" width="250">



>* Regularisierung verhindert Auswendiglernen der Trainingsdaten
>* Modelle lernen robustere Muster für neue Daten

>* Gewichtsstrafen fördern glatte, robuste Entscheidungen
>* Dropout, kleinere Modelle und Datenvielfalt helfen

>* Early Stopping schützt vor Überanpassung
>* Validierungsdaten zeigen echte Generalisierung



In [ ]:
#@title Python-Code - Regularisierung gegen Überanpassung

# Wir vergleichen Regularisierung gegen Überanpassung.
# L2 bestraft große Gewichte im Modell.
# Validierungsgenauigkeit zeigt bessere Verallgemeinerung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Ein kleiner Datensatz mit Rauschen macht Überanpassung sichtbar.
features, labels = make_moons(n_samples=500, noise=0.28, random_state=42)

# Die Aufteilung trennt Lernen und ehrliche Prüfung.
X_train, X_valid, y_train, y_valid = train_test_split(
    features, labels, test_size=0.35, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if X_train_scaled.shape[1] != 2:
    raise ValueError("Erwartet werden genau zwei Eingabemerkmale.")

# Ohne Regularisierung darf das Netz sehr flexible Gewichte lernen.
model_without_l2 = MLPClassifier(
    hidden_layer_sizes=(40,), alpha=0.0, max_iter=800, random_state=42
)
model_without_l2.fit(X_train_scaled, y_train)

# Mit L2 werden große Gewichte während des Trainings bestraft.
model_with_l2 = MLPClassifier(
    hidden_layer_sizes=(40,), alpha=0.08, max_iter=800, random_state=42
)
model_with_l2.fit(X_train_scaled, y_train)

# Wir messen Training und Validierung getrennt.
train_without = accuracy_score(y_train, model_without_l2.predict(X_train_scaled))
valid_without = accuracy_score(y_valid, model_without_l2.predict(X_valid_scaled))
train_with = accuracy_score(y_train, model_with_l2.predict(X_train_scaled))
valid_with = accuracy_score(y_valid, model_with_l2.predict(X_valid_scaled))

# Die Gewichtsnorm zeigt die Wirkung der L2-Strafe.
norm_without = sum(np.linalg.norm(weights) for weights in model_without_l2.coefs_)
norm_with = sum(np.linalg.norm(weights) for weights in model_with_l2.coefs_)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Ohne L2: Training {train_without:.2f}, Validierung {valid_without:.2f}")
print(f"Mit L2:  Training {train_with:.2f}, Validierung {valid_with:.2f}")
print(f"Gewichtsnorm ohne/mit L2: {norm_without:.1f} / {norm_with:.1f}")

# Das Diagramm vergleicht die entscheidenden Genauigkeiten.
labels_for_plot = ["Ohne L2 Training", "Ohne L2 Validierung", "Mit L2 Training", "Mit L2 Validierung"]
values_for_plot = [train_without, valid_without, train_with, valid_with]
colors = ["tab:blue", "tab:orange", "tab:blue", "tab:orange"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels_for_plot, values_for_plot, color=colors)
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Genauigkeit")
ax.set_title("Regularisierung kann die Validierungsleistung stabilisieren")
ax.tick_params(axis="x", rotation=20)
plt.show()



## **3. CNN Bausteine**

### **3.1. Faltung verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_01.jpg?v=1787654574" width="250">



>* Faltungen erkennen lokale Muster mit Filtern.
>* Filter betrachten kleine Nachbarschaften statt alles.

>* Geteilte Filter sparen viele Lernparameter
>* Muster werden ortsunabhängig erkannt

>* Feature Maps zeigen starke Filterreaktionen
>* Viele Filter lernen zunehmend abstrakte Merkmale



In [ ]:
#@title Python-Code - Faltung verstehen

# Diese Demo zeigt eine kleine Faltung.
# Ein Kernel sucht ein lokales Muster.
# Die Feature Map macht Treffer sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Dieses synthetische Bild enthält eine helle senkrechte Kante.
image = np.array(
    [[0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5]],
    dtype=float,
)

# Dieser Kernel reagiert stark auf senkrechte Helligkeitswechsel.
kernel = np.array(
    [[-1, 0, 1],
     [-1, 0, 1],
     [-1, 0, 1]],
    dtype=float,
)

# Die Ausgabe ist kleiner, weil kein Padding verwendet wird.
output_height = image.shape[0] - kernel.shape[0] + 1
output_width = image.shape[1] - kernel.shape[1] + 1
feature_map = np.zeros((output_height, output_width), dtype=float)

# Das Fenster gleitet über alle gültigen Positionen.
for row in range(output_height):
    for col in range(output_width):
        patch = image[row:row + 3, col:col + 3]
        feature_map[row, col] = np.sum(patch * kernel)

# Diese Prüfungen machen die erwarteten Formen explizit.
if feature_map.shape != (2, 4):
    raise ValueError("Die Feature-Map-Form ist unerwartet.")

# Kurze Ausgaben verbinden Rechnung und Tensorformen.
print(f"Eingabeform: {image.shape}")
print(f"Kernelform: {kernel.shape}")
print(f"Feature-Map-Form: {feature_map.shape}")
print(f"Stärkste Reaktion: {feature_map.max():.1f}")

# Die Heatmap zeigt, wo der Kernel das Muster findet.
fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(feature_map, cmap="viridis")
ax.set_title("Feature Map einer einfachen Faltung")
ax.set_xlabel("Spaltenposition des Kernels")
ax.set_ylabel("Zeilenposition des Kernels")
fig.colorbar(im, ax=ax, label="Aktivierung")
plt.show()



### **3.2. Padding und Pooling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_02.jpg?v=1787654576" width="250">



>* Padding schützt Randinformationen bei Faltungen
>* Künstliche Ränder beeinflussen Aktivierungen gezielt

>* Pooling verdichtet Merkmalskarten und spart Rechenaufwand
>* Mehr Robustheit, aber weniger genaue Details

>* Padding und Pooling steuern Tensorgrößen.
>* Balance zwischen Details, Speicher und Effizienz.



In [ ]:
#@title Python-Code - Padding und Pooling

# Dieses Beispiel zeigt Padding und Pooling.
# Kleine Matrizen machen Tensorformen sichtbar.
# Die Ausgabe vergleicht Größen und Werte.

import numpy as np
import matplotlib.pyplot as plt

# Diese synthetische Merkmalskarte bleibt bewusst klein.
feature_map = np.array(
    [[0, 1, 2, 0], [1, 3, 4, 1], [0, 2, 5, 2], [1, 0, 2, 3]],
    dtype=float,
)

# Padding ergänzt außen einen künstlichen Nullrand.
padded_map = np.pad(feature_map, pad_width=1, mode="constant", constant_values=0)

# Max-Pooling fasst lokale Zweierbereiche zusammen.
pooled_map = np.zeros((2, 2), dtype=float)
for row in range(2):
    for col in range(2):
        window = feature_map[row * 2:row * 2 + 2, col * 2:col * 2 + 2]
        pooled_map[row, col] = np.max(window)

# Eine einfache Prüfung schützt vor Formfehlern.
if padded_map.shape != (6, 6):
    raise ValueError("Das Padding sollte eine Form von 6 mal 6 erzeugen.")

# Die wichtigsten Formen werden kurz ausgegeben.
print(f"Eingabeform: {feature_map.shape}")
print(f"Nach Padding: {padded_map.shape}")
print(f"Nach 2x2-Max-Pooling: {pooled_map.shape}")
print(f"Pooling-Werte: {pooled_map.astype(int).ravel().tolist()}")

# Das Bild zeigt Padding als Rand und Pooling als Zahlen.
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(padded_map, cmap="Blues", vmin=0, vmax=5)
ax.set_title("Padding-Rand und Max-Pooling-Ergebnis")

# Achsen beschreiben die Positionen in der gepaddeten Karte.
ax.set_xlabel("Spalte der gepaddeten Merkmalskarte")
ax.set_ylabel("Zeile der gepaddeten Merkmalskarte")
ax.set_xticks(range(padded_map.shape[1]))
ax.set_yticks(range(padded_map.shape[0]))

# Zahlen machen die künstlichen Randwerte sichtbar.
for row in range(padded_map.shape[0]):
    for col in range(padded_map.shape[1]):
        ax.text(col, row, int(padded_map[row, col]), ha="center", va="center")

# Der Pooling-Hinweis verbindet Werte mit der Verkleinerung.
ax.text(2.5, 5.7, f"Max-Pooling ergibt {pooled_map.shape}: {pooled_map.astype(int).ravel().tolist()}", ha="center")
plt.show()



### **3.3. Tensorformen planen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_03.jpg?v=1787654578" width="250">



>* Tensorformen durch jede Schicht bewusst verfolgen
>* Faltungsparameter bestimmen passende Ausgabedimensionen

>* Pooling spart Rechenaufwand, verliert aber Details
>* Tensorformen müssen zur Aufgabe passen

>* Flattening braucht korrekt berechnete Tensorgrößen
>* Kanäle sind gelernte Merkmalsdetektoren



In [ ]:
#@title Python-Code - Tensorformen planen

# Wir planen Tensorformen für einfache CNN-Bausteine.
# Faltung und Pooling verändern Höhe und Breite.
# Die Ausgabe zeigt passende Formen vorab.

import numpy as np
import matplotlib.pyplot as plt

# Diese Funktion berechnet eine räumliche Ausgabelänge.
def output_length(input_length, kernel_size, stride, padding):
    numerator = input_length + 2 * padding - kernel_size
    return numerator // stride + 1

# Wir beschreiben einen kleinen Bildstapel im NHWC-Format.
batch_size = 4
height = 28
width = 28
channels = 3

# Diese Schicht nutzt acht Filter der Größe drei.
conv_filters = 8
conv_kernel = 3
conv_stride = 1
conv_padding = 1

# Pooling fasst jeweils zwei mal zwei Werte zusammen.
pool_kernel = 2
pool_stride = 2
pool_padding = 0

# Die Faltung erhält hier Höhe und Breite.
conv_height = output_length(height, conv_kernel, conv_stride, conv_padding)
conv_width = output_length(width, conv_kernel, conv_stride, conv_padding)
conv_shape = (batch_size, conv_height, conv_width, conv_filters)

# Die Aktivierung ändert nur Werte, nicht die Form.
activation_shape = conv_shape
pool_height = output_length(conv_height, pool_kernel, pool_stride, pool_padding)
pool_width = output_length(conv_width, pool_kernel, pool_stride, pool_padding)

# Nach Pooling wird die räumliche Auflösung kleiner.
pool_shape = (batch_size, pool_height, pool_width, conv_filters)
flatten_features = pool_height * pool_width * conv_filters
flatten_shape = (batch_size, flatten_features)

# Eine kleine Prüfung schützt vor unpassenden Formen.
if conv_height <= 0 or conv_width <= 0:
    raise ValueError("Die Faltung erzeugt keine gültige räumliche Form.")

if pool_height <= 0 or pool_width <= 0:
    raise ValueError("Das Pooling erzeugt keine gültige räumliche Form.")

# Wir geben eine kurze Formbilanz aus.
print(f"Eingabe: {(batch_size, height, width, channels)}")
print(f"Nach Faltung: {conv_shape}")
print(f"Nach Aktivierung: {activation_shape}")
print(f"Nach Pooling: {pool_shape}")
print(f"Nach Flatten: {flatten_shape}")

# Die Grafik zeigt die räumliche Verkleinerung.
stage_names = ["Eingabe", "Faltung", "Pooling"]
spatial_sizes = [height * width, conv_height * conv_width, pool_height * pool_width]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(stage_names, spatial_sizes, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Räumliche Größe durch CNN-Bausteine")
ax.set_xlabel("Stufe")
ax.set_ylabel("Höhe mal Breite")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**MLP und Faltungen**</font>


In this lecture, you learned to:
- Trainieren ein kleines Zwei-Schichten-MLP mit NumPy auf einfachen Klassifikationsdaten. 
- Vergleichen Mini-Batches, Shuffling, SGD, Momentum, Regularisierung und Early Stopping. 
- Implementieren grundlegende Faltungs- und Poolingoperationen und planen Tensorformen. 

In the next Module (Module 15), we will go over 'TensorFlow Keras'